In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

# Load data
df = pd.read_csv("/kaggle/input/llm-classification-finetuning/train.csv")

# Combine prompt with responses for better semantic context
def combine_inputs(row):
    return (
        f"Prompt: {row['prompt']}\n"
        f"Response A: {row['response_a']}\n"
        f"Response B: {row['response_b']}"
    )

df['input_a'] = df['prompt'] + "\n" + df['response_a']
df['input_b'] = df['prompt'] + "\n" + df['response_b']

# Optional: only use responses (you can toggle between both setups)
# df['input_a'] = df['response_a']
# df['input_b'] = df['response_b']

# Encode preference into 0, 1, 2
label_map = {'A': 0, 'B': 1, 'tie': 2}
df['label'] = df[['winner_model_a', 'winner_model_b', 'winner_tie']].values.argmax(axis=1)

# Load sentence transformer model
embedder = SentenceTransformer('/kaggle/input/all-minilm-l6-v2/pytorch/default/1/all-MiniLM-L6-v2')

# Encode response_a and response_b
print("Encoding responses...")
emb_a = embedder.encode(df['input_a'].tolist(), batch_size=32, show_progress_bar=True)
emb_b = embedder.encode(df['input_b'].tolist(), batch_size=32, show_progress_bar=True)

# Combine embeddings into a single feature (e.g., diff + abs diff)
X = np.hstack([
    emb_a - emb_b,
    np.abs(emb_a - emb_b),
])

y = df['label'].values

# Train/test split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.1, random_state=42, stratify=y)

# Train XGBoost classifier
clf = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    use_label_encoder=False,
    n_estimators=100,
    max_depth=4,
    learning_rate=0.1
)

print("Training XGBoost...")
clf.fit(X_train, y_train)

# Evaluate
y_pred = clf.predict(X_val)
print(classification_report(y_val, y_pred, target_names=['A', 'B', 'tie']))


In [ ]:

# Load test data
test_df = pd.read_csv("/kaggle/input/llm-classification-finetuning/test.csv")

test_df['input_a'] = test_df['prompt'] + "\n" + test_df['response_a']
test_df['input_b'] = test_df['prompt'] + "\n" + test_df['response_b']

# Encode test
emb_a_test = embedder.encode(test_df['input_a'].tolist(), batch_size=32, show_progress_bar=True)
emb_b_test = embedder.encode(test_df['input_b'].tolist(), batch_size=32, show_progress_bar=True)

X_test = np.hstack([
    emb_a_test - emb_b_test,
    np.abs(emb_a_test - emb_b_test),
])

# Predict probabilities
probs = clf.predict_proba(X_test)

# Create submission
submission = pd.DataFrame({
    "id": test_df["id"],
    "winner_model_a": probs[:, 0],
    "winner_model_b": probs[:, 1],
    "winner_tie": probs[:, 2],
})

submission.to_csv("submission.csv", index=False)